# GLM-OCR from base: native OCR format and balanced curriculum

Stage 1 starts from `model/glm-ocr` with a fresh rank-16 joint vision/language adapter and trainable projector. Stage 2 continues the selected Stage 1 checkpoint with approximately 40% complex/suspect exposures. Both retain the same 400-image development holdout and the 14 excluded IDs. **The original joint notebook is unchanged.**

Run setup and preview cells, then the two marked training cells. Each stage has a maximum of three epochs, with validation-based early stopping. The v7 run directory deliberately prevents resuming the incompatible v2-v6 adapters. No training is launched by setup or preview cells.

## What the comparison found

- Saved v6 Stage 1 results fall from local proxy **0.720495** at base to **0.162738** at step 77, with repeated lines and extra text in predictions. Ordinary labels also regress; complexity alone does not explain the collapse.
- The original `glm_ocr` LLaMA-Factory template supervises `\n + Target + EOS` after a plain assistant header. V6 adds `/nothink` and thinking tags. V7 uses the original native OCR format through one shared encoder, masks all prompt/image tokens, and keeps text/image token types aligned. This follows the [official fine-tuning recipe](https://github.com/zai-org/GLM-OCR/blob/main/examples/finetune/README.md) and the vendored template.
- Original training uses a spatial resize before the default processor; its monitor instead overrides the processor's internal bounds. Those are different paths. V7 retains the curriculum notebook's correct shared spatial resize followed by default patch rounding for training, validation and test.
- V6 changes language LR from 2e-6 to 5e-6, removes all complex/suspect training labels in Stage 1, and cuts five image views to one. V7 restores the original Stage 1 learning rates and one clean plus four fresh augmented views. All label groups enter Stage 1 by default; `ordinary_only=True` is an optional ablation. Heuristic suspect flags no longer reduce loss by default.
- Trainer changes generation stop-token defaults in memory. V7 explicitly supplies identical stop tokens for validation and submission, and reports blank outputs and token-limit hits.
- Epochs are not comparable across recipes: old Stage 1 had 16,305 exposures / about 2,039 updates per epoch; v6 had 2,462 / 308. Setup prints the new budget. Three configured epochs are preserved, but each now has more updates than v6.
- Original Stage 1 uses a 100-image monitor; Stage 2 trains on 623 formerly held-out images and correctly compares candidates on its remaining 200-image monitor. Those scores cannot be directly compared to the curriculum's 400-image holdout. Reusing the original 823-image validation set after Stage 2 would include training data.
- The original export cell still references `full_validation_comparison`, although the preceding cell now produces `monitor_comparison`; a clean top-to-bottom execution fails there. Its comparison loader also returns the base model for an `incoming` candidate even when that candidate is a Stage 1 adapter. These are original-notebook issues recorded here without editing that notebook.

The formatting differences and saved regression are confirmed; their individual causal contributions are not isolated. The revised recipe needs training to establish accuracy. Scores remain the project's local Zindi proxy, not a verified exact leaderboard evaluator. Validation is a development set; no held-out label enters training.


In [ ]:
%pip install -r requirements.txt


In [ ]:
import gc, hashlib, inspect, json, math, os, random, re, sys, unicodedata
from pathlib import Path
from dataclasses import asdict

os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'
import albumentations, cv2, numpy as np, pandas as pd, torch, transformers, peft
from PIL import Image, ImageOps
from IPython.display import display
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import AutoProcessor, GlmOcrForConditionalGeneration, Trainer, TrainingArguments, TrainerCallback
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
assert (ROOT / 'resources/Train.csv').is_file(), 'Open this notebook with the project root as working directory.'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.barbados_ocr_augmentation import HistoricalHandwritingAugmentation
from src.transcription_metrics import TranscriptionMetrics, ZindiScore

NOTEBOOK_NAME = 'notebooks/glm_finetuning.ipynb'
BASE_MODEL = ROOT / 'model/glm-ocr'
IMAGES = ROOT / 'resources/images'
CONFIG = {
    'run_name': 'glm_ocr_from_base_curriculum_v7_native_ocr_balanced',
    'seed': 42, 'holdout_size': 400,
    'prompt': 'Text Recognition:',
    'min_spatial_pixels': 384 * 28 * 28,
    'max_spatial_pixels': 1536 * 28 * 28,
    'max_sequence_length': 2048, 'max_new_tokens': 128,
    'batch_size': 1, 'gradient_accumulation': 8,
    'lora_rank': 16, 'lora_alpha': 32, 'lora_dropout': 0.05,
    'language_layers': list(range(8, 16)),
    'checkpoints_per_epoch': 4,
    'word_normalizer': 12.0, 'character_normalizer': 55.0,
    'suspect_loss_weight': 1.0,
    'stages': {
        'stage1': {'epochs': 3, 'vision_lr': 3e-5, 'language_lr': 2e-6,
                   'projector_lr': 2e-6, 'warmup_ratio': 0.05,
                   'ordinary_only': False, 'augmented_views': 4,
                   'severity': 0.30, 'augmentation_probability': 0.90},
        'stage2': {'epochs': 3, 'vision_lr': 5e-6, 'language_lr': 2e-6,
                   'projector_lr': 5e-7, 'warmup_ratio': 0.03,
                   'severity': 0.25, 'augmentation_probability': 0.65,
                   'complex_fraction': 0.40, 'augmented_views': 4},
    },
}
RUN = ROOT / 'training_outputs' / CONFIG['run_name']
VERSIONS = {name: module.__version__ for name, module in
            [('torch', torch), ('transformers', transformers), ('peft', peft),
             ('albumentations', albumentations), ('opencv', cv2), ('numpy', np)]}
assert torch.cuda.is_available(), (
    'CUDA is required. Select the Barbados MEDUSA kernel, restart it, and rerun from the top.')
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(CONFIG['seed'])
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU: setup only')
print('Output:', RUN)
print('Versions:', VERSIONS)
display(pd.DataFrame(CONFIG['stages']).T)


## Inspect labels and freeze the split

Ordinary labels may still contain names, historical spellings, numbers, and normal punctuation. Carets, unusual symbols, and a configurable list of common historical abbreviations mark complex labels. Repeated whitespace, repeated placeholders and replacement characters are **suspect flags**, not confirmed errors. Suspect flags are audit metadata. Rows receive full loss weight by default in both stages; `suspect_loss_weight` is configurable for a separate ablation.

The split is stratified by label group, crop-height band and word-count band. Rare combinations are merged. No validation ID enters either stage. These random IDs do not prove writer/page independence; use page groups if that metadata becomes available.

The 14 user-specified bad-image IDs are removed before stratification and splitting. They appear in neither training stage nor validation. Their original rows are recorded in `excluded_labels.csv`; source images and Train.csv are not modified. This policy uses a new run directory to avoid resuming checkpoints trained on the previous split.


In [ ]:
# User-specified exclusions apply to BOTH training stages and validation.
EXCLUDED_IDS = {
    '79tMUVyfIdy3GzkG',
    'rh8o7bdCGOIBFPwH',
    'F8DYDDp2AvW9Dytw',
    'mfQxfOmeRmwBh0g8',
    'yNyf3Tp0zc7DFj5F',
    'JU7lRwk3jKkus24Z',
    'R6iYPb7MHFiHtXH6',
    'VmrEALeZiP1Y6nF9',
    't0UrASljcgzvBAnO',
    'KH5g92Q3DA5Bo6xi',
    'N78M3v6GKmUzF7sk',
    'WwlTCykxjP3c4kfo',
    'u3b4JNo5bqpfE7Js',
    'MfT9S5oghk9ywNSC',
}

ABBREVIATION_PATTERN = r"\b(?:admrs?|exers?|execs?|pson?s?|prsent?s?|sd|ye|wch|wth|thos|jno|wm|esqr|coll|capt)\b"
ORDINARY_PUNCTUATION = set(".,;:!?\"'()&-/")

def label_flags(text):
    unusual = ''.join(sorted({c for c in text if not
        (c.isalnum() or c.isspace() or c in ORDINARY_PUNCTUATION)}))
    complex_reasons, suspect_reasons = [], []
    if '^' in text: complex_reasons.append('superscript_caret')
    if unusual: complex_reasons.append('unusual_symbols:' + unusual)
    if re.search(ABBREVIATION_PATTERN, text, flags=re.I): complex_reasons.append('abbreviation_pattern')
    if re.search(r'\s{2,}', text): suspect_reasons.append('repeated_whitespace')
    if re.search(r'(?:\bx\b\s*){3,}|_{3,}|\ufffd', text): suspect_reasons.append('placeholder_pattern')
    return '|'.join(complex_reasons), '|'.join(suspect_reasons)

source_labels = pd.read_csv(ROOT / 'resources/Train.csv', dtype=str, keep_default_na=False)
assert source_labels.ID.is_unique
missing_exclusions = EXCLUDED_IDS - set(source_labels.ID)
assert not missing_exclusions, f'Excluded IDs missing from Train.csv: {sorted(missing_exclusions)}'
excluded_labels = source_labels.loc[source_labels.ID.isin(EXCLUDED_IDS)].copy()
all_labels = source_labels.loc[~source_labels.ID.isin(EXCLUDED_IDS)].reset_index(drop=True)
assert set(all_labels.ID).isdisjoint(EXCLUDED_IDS)
assert len(all_labels) + len(excluded_labels) == len(source_labels)
assert all_labels.Target.str.strip().ne('').all()
print(f'Excluded {len(excluded_labels)} specified IDs; {len(all_labels)} eligible labeled images remain.')
assert all((IMAGES / f'{i}.jpg').is_file() for i in all_labels.ID)
flags = all_labels.Target.map(label_flags)
all_labels['complex_reasons'] = flags.map(lambda x: x[0])
all_labels['suspect_reasons'] = flags.map(lambda x: x[1])
all_labels['label_group'] = np.select(
    [all_labels.suspect_reasons.ne(''), all_labels.complex_reasons.ne('')],
    ['suspect', 'complex'], default='ordinary')
all_labels['word_count'] = all_labels.Target.str.split().str.len()
sizes = []
for image_id in all_labels.ID:
    with Image.open(IMAGES / f'{image_id}.jpg') as image: sizes.append(image.size)
all_labels['width'] = [x[0] for x in sizes]
all_labels['height'] = [x[1] for x in sizes]
all_labels['height_band'] = pd.cut(all_labels.height, [0, 80, 200, np.inf], labels=['small', 'medium', 'large']).astype(str)
all_labels['length_band'] = pd.cut(all_labels.word_count, [0, 8, 13, np.inf], labels=['short', 'medium', 'long']).astype(str)
strata = all_labels.label_group + '/' + all_labels.height_band + '/' + all_labels.length_band
strata = strata.where(strata.map(strata.value_counts()) >= 12, all_labels.label_group + '/other')
strata = strata.where(strata.map(strata.value_counts()) >= 2, 'rare')
if strata.value_counts().min() < 2: strata = all_labels.label_group
train_ids, val_ids = train_test_split(all_labels.ID, test_size=CONFIG['holdout_size'],
                                    random_state=CONFIG['seed'], stratify=strata)
all_labels['split'] = np.where(all_labels.ID.isin(val_ids), 'validation', 'train')
train_df = all_labels[all_labels.split.eq('train')].sort_values('ID').reset_index(drop=True)
val_df = all_labels[all_labels.split.eq('validation')].sort_values('ID').reset_index(drop=True)
assert set(train_df.ID).isdisjoint(val_df.ID)
assert set(train_df.ID).isdisjoint(EXCLUDED_IDS) and set(val_df.ID).isdisjoint(EXCLUDED_IDS)
assert len(val_df) == CONFIG['holdout_size'] and len(train_df) + len(val_df) == len(all_labels)
assert (train_df.label_group == 'ordinary').any() and (train_df.label_group != 'ordinary').any()
display(pd.crosstab(all_labels.label_group, all_labels.split))
display(pd.crosstab(all_labels.height_band, all_labels.split))
display(all_labels.loc[all_labels.label_group.eq('suspect'), ['ID','Target','suspect_reasons']].head(15))


In [ ]:
def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''): digest.update(block)
    return digest.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def write_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')
    temporary.replace(path)

# Sources, package versions, data and base weights define a compatible run.
# Hashing weights is read-only; this cell does not load the model onto the GPU.
notebook = json.loads((ROOT / NOTEBOOK_NAME).read_text(encoding='utf-8'))
notebook_source = [''.join(c['source']) for c in notebook['cells'] if c['cell_type'] == 'code']
base_files = sorted(p for p in BASE_MODEL.iterdir() if p.suffix in {'.json', '.safetensors', '.jinja', '.model'})
assert any(p.suffix == '.safetensors' for p in base_files)
identity = {
    'config': CONFIG, 'versions': VERSIONS,
    'excluded_ids': sorted(EXCLUDED_IDS),
    'notebook_code': object_hash(notebook_source),
    'augmentation_code': file_hash(ROOT / 'src/barbados_ocr_augmentation.py'),
    'metrics_code': file_hash(ROOT / 'src/transcription_metrics.py'),
    'train_csv': file_hash(ROOT / 'resources/Train.csv'),
    'base_files': {p.name: file_hash(p) for p in base_files},
    'training_images': {i: file_hash(IMAGES / f'{i}.jpg') for i in tqdm(all_labels.ID, desc='Fingerprint images')},
    'split': object_hash(all_labels[['ID','split','label_group']].to_dict('records')),
}
def compatible_stopping_upgrade(previous, current, sources):
    # Reviewed orchestration-only upgrade: Stage 2 branches from Stage 1 best.
    # Preserve the existing run/data identity while rejecting every config, data,
    # runtime, augmentation, metric, base-weight, image, and split change.
    if previous['notebook_code'] != '2b9f38cb8cfe8367e8e09afa831630954e9b1c001131b2865794796e9b815aeb': return False
    reviewed_sources = [source for source in sources
                        if 'def compatible_stopping_upgrade' not in source
                        and not source.lstrip().startswith('%pip install ')]
    if object_hash(reviewed_sources) != '4633e89e93d6cc51659a17b9b293a29ed9068ae84cfe2ef1983d70991a136740': return False
    # The original exact runtime is no longer registered locally. A runtime-only
    # migration is allowed solely before the new Stage 2 branch is initialized.
    ignored = {'notebook_code', 'versions'}
    before = {k:v for k,v in previous.items() if k not in ignored}
    after = {k:v for k,v in current.items() if k not in ignored}
    if before != after: return False
    fresh_stage2 = RUN / 'stage2_from_stage1_best'
    if fresh_stage2.exists() and any(fresh_stage2.iterdir()): return False
    if previous.get('versions') != current.get('versions'):
        print('Recorded runtime migration for fresh Stage 2:')
        print('  previous:', previous.get('versions'))
        print('  current: ', current.get('versions'))
    return True

IDENTITY_KEY = object_hash(identity)
RUN.mkdir(parents=True, exist_ok=True)
manifest = RUN / 'run_identity.json'
if manifest.exists():
    saved_identity = json.loads(manifest.read_text())
    previous_key = object_hash(saved_identity['identity'])
    if previous_key != IDENTITY_KEY:
        assert compatible_stopping_upgrade(saved_identity['identity'], identity, notebook_source), (
            'Training settings/data/code/runtime changed incompatibly. Choose a new run_name.')
        backup = RUN / f'run_identity_before_early_stopping_{previous_key[:16]}.json'
        if not backup.exists(): write_json(backup, saved_identity)
        write_json(manifest, {'run_key':saved_identity['run_key'], 'identity_key':IDENTITY_KEY,
                             'identity':identity, 'upgraded_from':previous_key})
        print('Early-stopping upgrade accepted; saved checkpoints and run lineage preserved.')
    RUN_KEY = saved_identity['run_key']
else:
    assert not any(RUN.iterdir()), 'Nonempty run directory without an identity; choose a new run_name.'
    RUN_KEY = IDENTITY_KEY
    write_json(manifest, {'run_key':RUN_KEY, 'identity_key':IDENTITY_KEY, 'identity':identity})
excluded_labels.to_csv(RUN / 'excluded_labels.csv', index=False)
all_labels.to_csv(RUN / 'split_and_label_audit.csv', index=False)
val_df.to_csv(RUN / 'validation_manifest.csv', index=False)
print('Run identity:', RUN_KEY[:16])


## Shared image processing and fresh augmentation

Both paths use the same explicit spatial resize, followed by the base processor's default patch-grid rounding. Do **not** replace its internal `size` with the spatial limits: GLM applies a temporal factor there. Training alone applies augmentation **before** the shared resize. No offline augmented JPEG cache is used.

Each Stage 2 epoch includes every training ID at least once. Repetitions bring complex/suspect examples to approximately 40% of exposures; ordinary examples remain present. Suspect rows receive full weight by default. Every sampled occurrence has one clean view plus four augmented views; Stage 1 covers all training IDs unless the optional ordinary-only ablation is enabled. The dataset receives the epoch before iteration; worker count remains zero so that updates and reproducibility are straightforward.


In [ ]:
processor = AutoProcessor.from_pretrained(BASE_MODEL, local_files_only=True, use_fast=False)
# Trainer may overwrite model.generation_config from tokenizer defaults. Pin the
# same stopping rules for incoming, in-memory, reloaded and submission models.
base_generation = json.loads((BASE_MODEL / 'generation_config.json').read_text(encoding='utf-8'))
base_eos = base_generation['eos_token_id']
base_eos = [base_eos] if isinstance(base_eos, int) else list(base_eos)
GENERATION_EOS_IDS = sorted(set(base_eos + [processor.tokenizer.eos_token_id]))
METRICS = TranscriptionMetrics(score=ZindiScore(
    word_normalizer=CONFIG['word_normalizer'], character_normalizer=CONFIG['character_normalizer']))

def load_rgb(image_id):
    with Image.open(IMAGES / f'{image_id}.jpg') as image:
        return ImageOps.exif_transpose(image).convert('RGB')

def resize_spatial(image):
    image = image.convert('RGB')
    for bound, comparison in [(CONFIG['max_spatial_pixels'], lambda area, x: area > x),
                              (CONFIG['min_spatial_pixels'], lambda area, x: area < x)]:
        if comparison(image.width * image.height, bound):
            scale = math.sqrt(bound / (image.width * image.height))
            image = image.resize((max(1, int(image.width * scale)), max(1, int(image.height * scale))),
                                 resample=Image.Resampling.BICUBIC)
    return image

def encode_prompt(image):
    # Match the vendored LLaMA-Factory glm_ocr template exactly. Its prompt
    # ends at <|assistant|>; the single newline belongs to the response.
    # The base chat template's assistant messages inject thinking markup.
    text = ('[gMASK]<sop><|user|>\n'
            '<|begin_of_image|><|image|><|end_of_image|>'
            + CONFIG['prompt'] + '<|assistant|>')
    batch = processor(text=[text], images=[image], add_special_tokens=False,
                      return_tensors='pt')
    batch.pop('token_type_ids', None)
    assert int(batch['input_ids'][0, -1]) == processor.tokenizer.convert_tokens_to_ids('<|assistant|>')
    assert 'mm_token_type_ids' in batch, 'Use a GLM-OCR processor that returns multimodal token types.'
    image_token_id = processor.tokenizer.convert_tokens_to_ids('<|image|>')
    assert torch.equal(batch['mm_token_type_ids'].eq(1), batch['input_ids'].eq(image_token_id))
    return batch


def stable_seed(*parts):
    return int(object_hash([CONFIG['seed'], *parts])[:8], 16)

class CurriculumDataset(Dataset):
    def __init__(self, frame, stage):
        self.frame = frame.reset_index(drop=True).copy()
        self.stage = stage; self.settings = CONFIG['stages'][stage]
        self.augmentation = HistoricalHandwritingAugmentation(
            severity=self.settings['severity'], probability=self.settings['augmentation_probability'])
        self.set_epoch(0)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        rng = np.random.default_rng(stable_seed(self.stage, self.epoch, 'sampling'))
        ordinary = np.flatnonzero(self.frame.label_group.eq('ordinary'))
        hard = np.flatnonzero(self.frame.label_group.ne('ordinary'))
        if self.stage == 'stage1':
            indices = ordinary.copy() if self.settings['ordinary_only'] else np.arange(len(self.frame))
        else:
            fraction = self.settings['complex_fraction']
            assert 0 < fraction < 1 and len(ordinary) and len(hard)
            total = math.ceil(max(len(ordinary) / (1 - fraction), len(hard) / fraction))
            hard_count = max(len(hard), round(total * fraction))
            ordinary_count = max(len(ordinary), total - hard_count)
            def cover_and_repeat(pool, count):
                return np.concatenate([pool, rng.choice(pool, count - len(pool), replace=True)])
            indices = np.concatenate([cover_and_repeat(ordinary, ordinary_count), cover_and_repeat(hard, hard_count)])
        # Each occurrence gets one guaranteed clean view and fresh augmented views.
        self.views_per_occurrence = 1 + int(self.settings['augmented_views'])
        assert self.views_per_occurrence >= 1
        indices = np.repeat(indices, self.views_per_occurrence)
        variants = np.tile(np.arange(self.views_per_occurrence), len(indices) // self.views_per_occurrence)
        order = rng.permutation(len(indices))
        self.indices = indices[order]
        self.variants = variants[order]
        self.exposures = self.frame.iloc[self.indices].label_group.value_counts().to_dict()

    def __len__(self): return len(self.indices)

    def __getitem__(self, index):
        row = self.frame.iloc[int(self.indices[index])]
        image = load_rgb(row.ID)
        if self.variants[index] > 0:
            image = Image.fromarray(self.augmentation.apply(
                np.asarray(image), seed=stable_seed(self.stage, self.epoch, row.ID, int(index))))
        return {'ID': row.ID, 'image': resize_spatial(image), 'Target': row.Target,
                'loss_weight': CONFIG['suspect_loss_weight'] if row.label_group == 'suspect' else 1.0}

class OCRCollator:
    def __call__(self, examples):
        assert len(examples) == 1, 'This memory-conscious collator requires batch_size=1.'
        row = examples[0]
        batch = encode_prompt(row['image'])
        eos_token_id = processor.tokenizer.eos_token_id
        assert eos_token_id is not None, 'The GLM tokenizer must define an EOS token.'
        response_ids = processor.tokenizer.encode(
            '\n' + str(row['Target']), add_special_tokens=False) + [eos_token_id]
        response_ids = torch.tensor([response_ids], dtype=batch['input_ids'].dtype)
        prompt_length = batch['input_ids'].shape[1]
        batch['input_ids'] = torch.cat([batch['input_ids'], response_ids], dim=1)
        batch['attention_mask'] = torch.cat(
            [batch['attention_mask'], torch.ones_like(response_ids)], dim=1)
        # GLM-OCR uses this mask to distinguish image tokens (1) from text tokens (0).
        # The appended target is text, so extend the mask with zeros to keep every
        # token-aligned tensor the same length.
        if 'mm_token_type_ids' in batch:
            batch['mm_token_type_ids'] = torch.cat(
                [batch['mm_token_type_ids'], torch.zeros_like(response_ids)], dim=1)
        assert batch['input_ids'].shape == batch['attention_mask'].shape
        if 'mm_token_type_ids' in batch:
            assert batch['input_ids'].shape == batch['mm_token_type_ids'].shape
        assert batch['input_ids'].shape[1] <= CONFIG['max_sequence_length'], (
            f"Sequence too long for {row['ID']}; refusing to truncate image tokens or target text.")
        labels = torch.full_like(batch['input_ids'], -100)
        labels[:, prompt_length:] = response_ids
        assert torch.equal(labels[:, prompt_length:], response_ids)
        assert int(labels[0, -1]) == eos_token_id
        batch['labels'] = labels
        batch['sample_weight'] = torch.tensor(float(row['loss_weight']))
        return batch

collator = OCRCollator()
for stage in CONFIG['stages']:
    preview_dataset = CurriculumDataset(train_df, stage)
    print(stage, 'exposures/epoch:', len(preview_dataset), preview_dataset.exposures)
print('Processor internal size (base defaults):', processor.image_processor.size)


In [ ]:
# CPU-only preprocessing and label-mask smoke check: no model or training.
small_id = train_df.loc[train_df.height.idxmin(), 'ID']
large_id = train_df.loc[train_df.height.idxmax(), 'ID']
checks = []
for image_id in [small_id, large_id]:
    row = train_df.set_index('ID').loc[image_id]
    image = resize_spatial(load_rgb(image_id))
    batch = collator([{'ID':image_id, 'image':image, 'Target':row.Target, 'loss_weight':1.0}])
    inference_batch = encode_prompt(image)
    assert torch.equal(batch['image_grid_thw'], inference_batch['image_grid_thw'])
    assert torch.equal(batch['pixel_values'], inference_batch['pixel_values'])
    prefix_length = inference_batch['input_ids'].shape[1]
    assert torch.equal(batch['input_ids'][:, :prefix_length], inference_batch['input_ids'])
    assert batch['labels'][:, :prefix_length].eq(-100).all()
    assert batch['mm_token_type_ids'][:, prefix_length:].eq(0).all()
    response = processor.tokenizer.encode('\n' + row.Target, add_special_tokens=False) + [processor.tokenizer.eos_token_id]
    assert batch['labels'][0, prefix_length:].tolist() == response
    checks.append({'ID':image_id,'spatial_size':image.size,
                   'visual_tokens':int(batch['image_grid_thw'].prod().item() // 4),
                   'sequence_tokens':batch['input_ids'].shape[1],
                   'supervised_tokens':int((batch['labels'] != -100).sum())})
display(pd.DataFrame(checks))
example_id = train_df.loc[train_df.label_group.eq('ordinary')].iloc[0].ID
original = np.asarray(load_rgb(example_id))
for severity in [0, 0.3, 0.6]:
    variant = HistoricalHandwritingAugmentation(severity, probability=1).apply(original, seed=42)
    thumbnail = Image.fromarray(variant); thumbnail.thumbnail((1200, 180))
    print('Severity:', severity); display(thumbnail)

# Cheap tokenization only: no image tensor or model pass for this full label audit.
target_lengths = all_labels.Target.map(lambda text: len(processor.tokenizer.encode(
    '\n' + text, add_special_tokens=False)) + 1)
assert target_lengths.max() <= CONFIG['max_new_tokens'], 'Generation limit is shorter than some labeled responses.'
print('Longest labeled response including newline/EOS:', int(target_lengths.max()))
print('Optimizer steps per epoch:', {stage: math.ceil(len(CurriculumDataset(train_df, stage)) /
    CONFIG['gradient_accumulation']) for stage in CONFIG['stages']})


## Checkpoints, early stopping and resuming

Three epochs per stage is now a maximum. Both stages stop after the first strict decrease in validation Zindi proxy at a saved checkpoint (`EARLY_STOPPING`: patience 1, minimum decrease 0). Stage 1 retains quarter-epoch checks; Stage 2 checks every 0.1 epoch. Equal scores do not trigger stopping. A single decrease may be noise rather than overfitting; patience can be configured before a new run.

The checkpoint is saved before evaluation. On resume, saved score history is checked before further updates. `best.json` records the highest-scoring candidate. Stage 2 now starts from that Stage 1 best checkpoint, saves every 0.1 epoch, and writes to the separate `stage2_from_stage1_best` directory, preserving the earlier Stage 2 experiment. Stage 2 retains best-score final selection, including its incoming model.

For an already manually stopped Stage 1, rerun setup/definitions and then run the Stage 2 cell directly. It finalizes Stage 1 from saved progress and writes `completed.json`; Stage 1 does not need to reach three epochs. If validation was interrupted, only the latest saved checkpoint's validation is completed first. Unsaved updates cannot be recovered. An existing completion/handoff is reused so reruns cannot silently change Stage 2's starting model.

This specific stopping/handoff upgrade accepts the previous v7 identity while preserving its run key. Dataset, split, configuration, image processing and runtime identity still have to match. Checkpoint directories and model weights are preserved.


In [ ]:
VISION_TARGETS = [f'visual.blocks.{layer}.{name}' for layer in range(24)
                  for name in ('attn.qkv','attn.proj','mlp.gate_proj','mlp.up_proj','mlp.down_proj')]
LANGUAGE_TARGETS = [f'language_model.layers.{layer}.{name}' for layer in CONFIG['language_layers']
                    for name in ('self_attn.q_proj','self_attn.k_proj','self_attn.v_proj','self_attn.o_proj',
                                 'mlp.gate_proj','mlp.up_proj','mlp.down_proj')]

BASE_SPEC = {'kind':'base', 'path':None}

def model_path(spec):
    if spec['kind'] == 'base': return BASE_MODEL
    path = RUN / spec['path']
    assert (path / 'adapter_config.json').is_file(), path
    return path

def load_model(spec, training=False):
    assert torch.cuda.is_available(), 'Training and model inference require a CUDA GPU.'
    seed_everything(CONFIG['seed'])
    base = GlmOcrForConditionalGeneration.from_pretrained(
        BASE_MODEL, local_files_only=True, dtype=DTYPE, device_map={'':0}, attn_implementation='sdpa')
    if spec['kind'] == 'adapter':
        model = PeftModel.from_pretrained(base, model_path(spec), is_trainable=training)
    elif training:
        names = set(dict(base.named_modules()))
        targets = VISION_TARGETS + LANGUAGE_TARGETS
        # Transformers versions may fuse language gate/up projections. Adapt
        # the fresh LoRA targets to actual modules instead of skipping them.
        for layer in CONFIG['language_layers']:
            prefix = f'language_model.layers.{layer}.mlp.'
            if any(n.endswith(prefix + 'gate_up_proj') for n in names):
                targets = [t for t in targets if t not in {prefix + 'gate_proj', prefix + 'up_proj'}]
                targets.append(prefix + 'gate_up_proj')
        missing = [name for name in targets if not any(n.endswith(name) for n in names)]
        assert not missing, f'LoRA modules missing: {missing}'
        print('Resolved LoRA modules:', len(targets))
        model = get_peft_model(base, LoraConfig(
            task_type=TaskType.CAUSAL_LM, r=CONFIG['lora_rank'], lora_alpha=CONFIG['lora_alpha'],
            lora_dropout=CONFIG['lora_dropout'], target_modules=targets,
            modules_to_save=['visual.merger'], bias='none'))
    else:
        model = base
    if training:
        model.config.use_cache = False
        model.enable_input_require_grads()
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
        model.train()
    else: model.eval()
    return model

def predict(model, image_id, beams=1, return_details=False):
    image = resize_spatial(load_rgb(image_id))
    inputs = encode_prompt(image).to(next(model.parameters()).device)
    options = {'max_new_tokens':CONFIG['max_new_tokens'], 'num_beams':beams,
               'do_sample':False, 'use_cache':True,
               'eos_token_id':GENERATION_EOS_IDS,
               'pad_token_id':processor.tokenizer.pad_token_id}
    if beams > 1: options.update(early_stopping=True, length_penalty=1.0)
    with torch.inference_mode(): output = model.generate(**inputs, **options)
    generated = output[0, inputs['input_ids'].shape[1]:]
    eos_ids = GENERATION_EOS_IDS
    text = unicodedata.normalize('NFC', processor.decode(
        generated, skip_special_tokens=True, clean_up_tokenization_spaces=False)).strip()
    details = {'Prediction':text, 'generated_tokens':len(generated),
               'hit_token_limit':len(generated) >= CONFIG['max_new_tokens'] and int(generated[-1]) not in eos_ids}
    return details if return_details else text


def evaluate(model, frame, output_stem, beams=1):
    # Evaluation must not consume the training/dropout/augmentation RNG stream.
    python_state = random.getstate(); numpy_state = np.random.get_state()
    was_training = model.training
    try:
        with torch.random.fork_rng(devices=[0]):
            model.eval()
            predictions = [predict(model, row.ID, beams, return_details=True) for row in
                           tqdm(frame.itertuples(), total=len(frame), desc='Validation')]
    finally:
        model.train(was_training); random.setstate(python_state); np.random.set_state(numpy_state)
    results = frame[['ID','Target','label_group','height_band']].copy()
    for column in ['Prediction', 'generated_tokens', 'hit_token_limit']:
        results[column] = [row[column] for row in predictions]
    results.to_csv(str(output_stem) + '_predictions.csv', index=False)
    metrics = METRICS(results.Target, results.Prediction)
    metrics['token_limit_rate'] = float(results.hit_token_limit.mean())
    metrics['blank_rate'] = float(results.Prediction.str.strip().eq('').mean())
    metrics['exact_match_rate'] = float((results.Target == results.Prediction).mean())
    slices = []
    for column in ['label_group','height_band']:
        for value, group in results.groupby(column):
            slices.append({'grouping':column,'group':value,'rows':len(group),
                           **METRICS(group.Target, group.Prediction)})
    pd.DataFrame(slices).to_csv(str(output_stem) + '_slices.csv', index=False)
    write_json(str(output_stem) + '_metrics.json', metrics)
    return metrics

class CurriculumTrainer(Trainer):
    def __init__(self, *args, stage_settings, **kwargs):
        self.stage_settings = stage_settings
        super().__init__(*args, **kwargs)
        # Batch size is one. Weight the sample's mean supervised-token loss.
        self.model_accepts_loss_kwargs = False

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs = dict(inputs); weight = inputs.pop('sample_weight')
        loss, outputs = super().compute_loss(model, inputs, return_outputs=True)
        loss = loss * weight.to(loss.device)
        return (loss, outputs) if return_outputs else loss

    def create_optimizer(self):
        if self.optimizer is None:
            groups = {'vision':[], 'language':[], 'projector':[]}
            for name, parameter in self.model.named_parameters():
                if not parameter.requires_grad: continue
                if '.lora_' in name and 'visual.blocks.' in name: key = 'vision'
                elif '.lora_' in name and 'language_model.layers.' in name: key = 'language'
                elif 'visual.merger' in name: key = 'projector'
                else: raise AssertionError('Unexpected trainable parameter: ' + name)
                groups[key].append(parameter)
            assert all(groups.values())
            self.optimizer = torch.optim.AdamW([
                {'params':params, 'lr':self.stage_settings[key + '_lr'],
                 'weight_decay':0.03 if key == 'projector' else 0.01, 'name':key}
                for key, params in groups.items()])
            print('Trainable parameters:', {key:sum(p.numel() for p in ps) for key,ps in groups.items()})
        return self.optimizer


In [ ]:
# Stop on the first strict decrease between successive validation scores.
# Evaluations occur at the configured saved epoch fractions, not every batch.
EARLY_STOPPING = {'patience': 1, 'min_decrease': 0.0}
STAGE1_HANDOFF_SELECTION = 'best'
CHECKPOINTS_PER_EPOCH = {'stage1':CONFIG['checkpoints_per_epoch'], 'stage2':10}
STAGE_DIRECTORIES = {'stage1':'stage1', 'stage2':'stage2_from_stage1_best'}

def stage_dir_for(stage):
    return RUN / STAGE_DIRECTORIES[stage]

def score_decreased(history):
    patience = int(EARLY_STOPPING['patience'])
    minimum = float(EARLY_STOPPING['min_decrease'])
    assert patience >= 1 and minimum >= 0
    rows = sorted(history, key=lambda row: row['step'])
    scores = [float(row['metrics']['zindi_score_proxy']) for row in rows]
    assert all(math.isfinite(score) for score in scores), 'Non-finite validation score; inspect metrics.'
    if len(scores) <= patience: return False
    return all(scores[i] < scores[i - 1] - minimum
               for i in range(len(scores) - patience, len(scores)))


def finish_saved_stage(stage, selection='last', reason='manual_stop'):
    """Finalize saved progress without performing any further training updates."""
    assert selection in {'last', 'best'}
    stage_dir = stage_dir_for(stage)
    stage_identity = stage_dir / 'stage_identity.json'
    assert stage_identity.is_file(), f'No initialized {stage} run.'
    saved = json.loads(stage_identity.read_text())
    expected = object_hash({'run_key':RUN_KEY, 'stage':stage, 'incoming':saved['incoming']})
    assert saved['stage_key'] == expected, 'Stage belongs to another run.'
    completed = stage_dir / 'completed.json'
    if completed.exists():
        result = json.loads(completed.read_text())
        model_path(result['spec'])
        return result
    ready = complete_checkpoints(stage_dir / 'checkpoints')
    assert ready, f'No complete {stage} checkpoint exists; run that stage until a checkpoint is saved.'
    latest = ready[-1]
    latest_step = int(latest.name.split('-')[-1])
    history_path = stage_dir / 'history.json'
    assert history_path.exists(), 'Restore the missing validation history.'
    history = json.loads(history_path.read_text())
    if latest_step not in {row['step'] for row in history}:
        # An interruption during validation must not silently lose the last checkpoint.
        print('Finishing validation of saved checkpoint; no training:', latest.name)
        model = load_model({'kind':'adapter', 'path':latest.relative_to(RUN).as_posix()})
        try:
            evaluate_checkpoint(model, stage_dir, latest,
                                json.loads((latest / 'trainer_state.json').read_text()))
        finally:
            del model; gc.collect(); torch.cuda.empty_cache()
        history = json.loads(history_path.read_text())
    available = {path.relative_to(RUN).as_posix() for path in ready}
    candidates = [row for row in history if row['step'] <= latest_step and
                  (row['step'] == 0 or row['spec']['path'] in available)]
    assert candidates and all(math.isfinite(float(row['metrics']['zindi_score_proxy'])) for row in candidates)
    chosen = (next(row for row in candidates if row['step'] == latest_step) if selection == 'last'
              else max(candidates, key=lambda row:row['metrics']['zindi_score_proxy']))
    model_path(chosen['spec'])
    result = {**chosen, 'completion_reason':reason, 'handoff_selection':selection,
              'stopped_step':latest_step, 'stopped_epoch':json.loads(
                  (latest / 'trainer_state.json').read_text())['epoch']}
    write_json(completed, result)
    print(f'{stage} finalized at saved step {latest_step}; selected:', result['spec'])
    return result


def checkpoint_is_complete(path):
    """Inspect saved state without loading optimizer tensors or using the GPU."""
    import zipfile
    from safetensors import SafetensorError, safe_open
    path = Path(path)
    required = ['adapter_config.json', 'adapter_model.safetensors', 'trainer_state.json',
                'optimizer.pt', 'scheduler.pt', 'rng_state.pth']
    if DTYPE == torch.float16: required.append('scaler.pt')
    if not all((path / name).is_file() and (path / name).stat().st_size > 0 for name in required):
        return False
    try:
        state = json.loads((path / 'trainer_state.json').read_text())
        json.loads((path / 'adapter_config.json').read_text())
        if int(state['global_step']) != int(path.name.split('-')[-1]): return False
        with safe_open(str(path / 'adapter_model.safetensors'), framework='pt', device='cpu') as weights:
            if not list(weights.keys()): return False
        return all(zipfile.is_zipfile(path / name) for name in required
                   if name.endswith(('.pt', '.pth')))
    except (ValueError, KeyError, OSError, RuntimeError, SafetensorError):
        return False


def complete_checkpoints(checkpoint_dir):
    if not checkpoint_dir.exists(): return []
    paths = sorted((p for p in checkpoint_dir.glob('checkpoint-*')
                    if p.is_dir() and p.name.split('-')[-1].isdigit()),
                   key=lambda p:int(p.name.split('-')[-1]))
    ready = []
    for path in paths:
        if checkpoint_is_complete(path): ready.append(path)
        else: print('Ignoring incomplete checkpoint:', path.name)
    return ready


def save_metric_history(stage_dir, history):
    """JSON is authoritative; rebuild best selection and a flat CSV from it."""
    history = sorted(history, key=lambda row:row['step'])
    write_json(stage_dir / 'history.json', history)
    winner = max(history, key=lambda row:row['metrics']['zindi_score_proxy'])
    write_json(stage_dir / 'best.json', winner)
    flat = []
    for row in history:
        flat.append({'step':row['step'], 'epoch':row['epoch'],
                     'candidate_kind':row['spec']['kind'], 'checkpoint':row['spec']['path'],
                     'last_logged_loss':row.get('last_logged_loss'),
                     'loss_log_step':row.get('loss_log_step'),
                     'last_logged_learning_rate':row.get('last_logged_learning_rate'),
                     **row['metrics']})
    temporary = stage_dir / 'metrics.csv.tmp'
    pd.DataFrame(flat).to_csv(temporary, index=False)
    temporary.replace(stage_dir / 'metrics.csv')
    return winner


def evaluate_checkpoint(model, stage_dir, checkpoint, state):
    step = int(state['global_step'])
    stem = stage_dir / 'validation' / f'step-{step}'
    stem.parent.mkdir(exist_ok=True)
    metrics = evaluate(model, val_df, stem)
    loss_logs = [entry for entry in state.get('log_history', []) if 'loss' in entry]
    latest = loss_logs[-1] if loss_logs else {}
    row = {'step':step, 'epoch':float(state['epoch']),
           'spec':{'kind':'adapter','path':checkpoint.relative_to(RUN).as_posix()},
           'metrics':metrics, 'last_logged_loss':latest.get('loss'),
           'loss_log_step':latest.get('step'),
           'last_logged_learning_rate':latest.get('learning_rate')}
    history = json.loads((stage_dir / 'history.json').read_text())
    history = [entry for entry in history if entry['step'] != step] + [row]
    winner = save_metric_history(stage_dir, history)
    print(f'Epoch {row["epoch"]:.4f}, step {step}:', metrics)
    print('Best local Zindi proxy:', winner['metrics']['zindi_score_proxy'])


class StageProgress(TrainerCallback):
    def __init__(self, dataset, stage_dir):
        self.dataset = dataset; self.stage_dir = stage_dir
        self.next_save_epoch = 1 / CHECKPOINTS_PER_EPOCH[self.dataset.stage]

    def on_train_begin(self, args, state, control, **kwargs):
        # Trainer has restored the previous global step/epoch by this point.
        checkpoints_per_epoch = CHECKPOINTS_PER_EPOCH[self.dataset.stage]
        self.next_save_epoch = (math.floor(float(state.epoch or 0) * checkpoints_per_epoch + 1e-9) + 1) / checkpoints_per_epoch

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.dataset.set_epoch(int(state.epoch or 0))
        print('Epoch', self.dataset.epoch + 1, 'sampling:', self.dataset.exposures)

    def on_step_end(self, args, state, control, **kwargs):
        # Save at the first optimizer boundary reaching .25/.50/.75/1.00, etc.
        # Absolute epoch thresholds avoid accumulated drift from a fixed step interval.
        epoch = float(state.epoch or 0)
        if epoch + 1e-9 >= self.next_save_epoch or state.global_step >= state.max_steps:
            control.should_save = True
            checkpoints_per_epoch = CHECKPOINTS_PER_EPOCH[self.dataset.stage]
            self.next_save_epoch = (math.floor(epoch * checkpoints_per_epoch + 1e-9) + 1) / checkpoints_per_epoch
        return control

    def on_save(self, args, state, control, model=None, **kwargs):
        checkpoint = Path(args.output_dir) / f'checkpoint-{int(state.global_step)}'
        assert checkpoint_is_complete(checkpoint), f'Checkpoint was not fully saved: {checkpoint}'
        # Model/optimizer/scheduler/RNG are already on disk BEFORE validation starts.
        evaluate_checkpoint(model, self.stage_dir, checkpoint, asdict(state))
        history = json.loads((self.stage_dir / 'history.json').read_text())
        if score_decreased(history):
            control.should_training_stop = True
            print('Early stopping: validation Zindi proxy decreased. Checkpoint is safely saved.')
        return control


def training_arguments(stage, dataset):
    settings = CONFIG['stages'][stage]
    return TrainingArguments(
        output_dir=str(stage_dir_for(stage) / 'checkpoints'),
        per_device_train_batch_size=1, gradient_accumulation_steps=CONFIG['gradient_accumulation'],
        num_train_epochs=settings['epochs'], learning_rate=settings['vision_lr'],
        warmup_ratio=settings['warmup_ratio'], lr_scheduler_type='cosine',
        weight_decay=0.01, max_grad_norm=1.0,
        bf16=DTYPE == torch.bfloat16, fp16=DTYPE == torch.float16,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant':False},
        # StageProgress requests saves at the configured epoch-fraction boundaries.
        save_strategy='no', save_total_limit=None, save_only_model=False, eval_strategy='no',
        logging_steps=20, report_to='none', remove_unused_columns=False,
        dataloader_num_workers=0, dataloader_pin_memory=False,
        seed=CONFIG['seed'], data_seed=CONFIG['seed'], optim='adamw_torch',
    )


def incoming_for(stage):
    if stage == 'stage1': return BASE_SPEC
    stage1_best = stage_dir_for('stage1') / 'best.json'
    if not stage1_best.exists():
        finish_saved_stage('stage1', selection='best', reason='manual_stop_for_stage2')
    selected = json.loads(stage1_best.read_text())
    model_path(selected['spec'])
    print('Stage 2 incoming model is Stage 1 best:', selected['spec'])
    return selected['spec']


def train_stage(stage):
    assert torch.cuda.is_available() and CONFIG['batch_size'] == 1
    stage_dir = stage_dir_for(stage); stage_dir.mkdir(exist_ok=True)
    incoming = incoming_for(stage)
    stage_key = object_hash({'run_key':RUN_KEY,'stage':stage,'incoming':incoming})
    identity_path = stage_dir / 'stage_identity.json'
    if identity_path.exists():
        assert json.loads(identity_path.read_text())['stage_key'] == stage_key
    else:
        assert not any(stage_dir.iterdir()), 'Stage files without identity; use a new run_name.'
        write_json(identity_path, {'stage_key':stage_key,'incoming':incoming})
    completed = stage_dir / 'completed.json'
    if completed.exists():
        winner = json.loads(completed.read_text())
        if stage == 'stage1' and STAGE1_HANDOFF_SELECTION == 'best':
            winner = json.loads((stage_dir / 'best.json').read_text())
        model_path(winner['spec'])
        print('Stage already complete; selected for handoff:', stage, winner)
        return winner

    checkpoint_dir = stage_dir / 'checkpoints'
    ready = complete_checkpoints(checkpoint_dir)
    last_checkpoint = str(ready[-1]) if ready else None
    history_path = stage_dir / 'history.json'
    if last_checkpoint: assert history_path.exists(), 'Restore the missing stage metric history.'
    if not history_path.exists():
        gc.collect(); torch.cuda.empty_cache()
        baseline_model = load_model(incoming)
        validation_dir = stage_dir / 'validation'; validation_dir.mkdir(exist_ok=True)
        try: metrics = evaluate(baseline_model, val_df, validation_dir / 'incoming')
        finally:
            del baseline_model; gc.collect(); torch.cuda.empty_cache()
        row = {'step':0,'epoch':0.0,'spec':incoming,'metrics':metrics}
        save_metric_history(stage_dir, [row])
    else:
        # Repairs an interruption between JSON, best selection and CSV writes.
        save_metric_history(stage_dir, json.loads(history_path.read_text()))

    # A stop during checkpoint validation leaves the training state intact.
    # Evaluate saved-but-unscored checkpoints before any further optimizer updates.
    scored_steps = {r['step'] for r in json.loads(history_path.read_text())}
    for checkpoint in ready:
        step = int(checkpoint.name.split('-')[-1])
        if step in scored_steps: continue
        print('Completing interrupted validation:', checkpoint.name)
        recovery_model = load_model({'kind':'adapter','path':checkpoint.relative_to(RUN).as_posix()})
        try:
            evaluate_checkpoint(recovery_model, stage_dir, checkpoint,
                                json.loads((checkpoint / 'trainer_state.json').read_text()))
        finally:
            del recovery_model; gc.collect(); torch.cuda.empty_cache()

    # Respect an already-observed decrease when resuming an interrupted run.
    if last_checkpoint and score_decreased(json.loads(history_path.read_text())):
        return finish_saved_stage(stage, selection=(STAGE1_HANDOFF_SELECTION if stage == 'stage1' else 'best'),
                                  reason='early_stopping')

    dataset = CurriculumDataset(train_df, stage)
    settings = CONFIG['stages'][stage]
    training_spec = ({'kind':'adapter','path':Path(last_checkpoint).relative_to(RUN).as_posix()}
                     if last_checkpoint else incoming)
    if last_checkpoint:
        state = json.loads((Path(last_checkpoint) / 'trainer_state.json').read_text())
        # A final checkpoint can be complete even if the process stopped before
        # writing the stage completion file. Do not train beyond its planned end.
        if int(state['global_step']) >= int(state['max_steps']):
            winner = save_metric_history(stage_dir, json.loads(history_path.read_text()))
            write_json(completed, winner)
            print('Recovered completed stage:', stage, winner)
            return winner
        print('Resuming model, optimizer, scheduler and RNG from:', last_checkpoint)
    else:
        print('No complete checkpoint yet; starting this stage from its incoming model.')
    gc.collect(); torch.cuda.empty_cache()
    model = load_model(training_spec, training=True)
    trainer = CurriculumTrainer(
        model=model, args=training_arguments(stage, dataset), stage_settings=settings,
        train_dataset=dataset, data_collator=collator, processing_class=processor,
        callbacks=[StageProgress(dataset, stage_dir)])
    write_json(stage_dir / 'training_plan.json', {
        'epochs':settings['epochs'],'exposures_per_epoch':len(dataset),'groups':dataset.exposures,
        'settings':settings,'resume_from':last_checkpoint,
        'checkpoint_interval_epochs':1 / CHECKPOINTS_PER_EPOCH[stage]})
    try:
        trainer.train(resume_from_checkpoint=last_checkpoint)
        # on_step_end also requests a complete final training-state checkpoint.
        final_step = int(trainer.state.global_step)
        final_checkpoint = checkpoint_dir / f'checkpoint-{final_step}'
        assert checkpoint_is_complete(final_checkpoint), 'Final checkpoint missing; stage remains resumable.'
        history = json.loads(history_path.read_text())
        if final_step not in {r['step'] for r in history}:
            evaluate_checkpoint(model, stage_dir, final_checkpoint, asdict(trainer.state))
        history = json.loads(history_path.read_text())
        if score_decreased(history):
            winner = finish_saved_stage(stage,
                selection=(STAGE1_HANDOFF_SELECTION if stage == 'stage1' else 'best'), reason='early_stopping')
        else:
            winner = save_metric_history(stage_dir, history)
            write_json(completed, winner)
    finally:
        del trainer, model; gc.collect(); torch.cuda.empty_cache()
    print('Selected:', stage, winner)
    return winner


## RUN STAGE 1 - starts from the original base model

Creates a fresh adapter and trains all label groups by default, with one clean and four augmented views per image. The optional `ordinary_only` ablation restricts training to ordinary labels. Reruns resume only this run's compatible checkpoints. This is a long-running cell.


In [ ]:
stage1_selection = train_stage('stage1')


## RUN STAGE 2 - continue after Stage 1 stops

Run this cell after setup and definitions, including when you manually interrupted Stage 1. An unfinished Stage 1 is finalized from its latest complete checkpoint without further training. Stage 2 loads that adapter and starts its own optimizer and learning-rate schedule. With the currently saved history, the latest Stage 1 checkpoint is step 2879 (proxy 0.86516); its separately recorded best is step 2303 (0.87963).

Stage 2 still trains on the entire training split with approximately 40% complex/suspect exposures and never trains on validation labels. It also stops on a validation-score decrease, retaining its best-scoring candidate for final selection.


In [ ]:
stage2_selection = train_stage('stage2')


## Select the final candidate

The Stage 2 history includes its incoming Stage 1 model, so the earlier model remains available if continuation fails to improve. Scores below are all on the same development holdout. Optional beam-four evaluation is disabled by default; turn it on only after training to assess whether it helps this particular adapter.


In [ ]:
results = []
for stage in ['stage1','stage2']:
    path = stage_dir_for(stage) / ('best.json' if stage == 'stage1' and STAGE1_HANDOFF_SELECTION == 'best' else 'completed.json')
    assert path.exists(), f'Complete {stage} first.'
    row = json.loads(path.read_text())
    results.append({'stage':stage, **row['spec'], **row['metrics']})
display(pd.DataFrame(results))
selected_record = json.loads((stage_dir_for('stage2') / 'completed.json').read_text())
FINAL_SPEC = selected_record['spec']
FINAL_BEAMS = 1
RUN_BEAM4_COMPARISON = False
if RUN_BEAM4_COMPARISON:
    model = load_model(FINAL_SPEC)
    comparison_dir = RUN / 'decoding_comparison'; comparison_dir.mkdir(exist_ok=True)
    try: beam_metrics = evaluate(model, val_df, comparison_dir / 'beam4', beams=4)
    finally:
        del model; gc.collect(); torch.cuda.empty_cache()
    if beam_metrics['zindi_score_proxy'] > selected_record['metrics']['zindi_score_proxy']:
        FINAL_BEAMS = 4
write_json(RUN / 'final_selection.json', {'run_key':RUN_KEY,'spec':FINAL_SPEC,'num_beams':FINAL_BEAMS})
print('Final model:', FINAL_SPEC, 'Beams:', FINAL_BEAMS)


## CREATE TEST SUBMISSION - inference only

Requires the final selection above. Cache identity includes the model contents, preprocessing/generation configuration, source code, package versions, and each input image hash. Failed/blank predictions are reported instead of silently submitted. Existing submissions from other notebooks are not touched.


In [ ]:
selection = json.loads((RUN / 'final_selection.json').read_text())
assert selection['run_key'] == RUN_KEY
spec = selection['spec']; beams = int(selection['num_beams'])
selected_path = model_path(spec)
weight_identity = ({p.name:file_hash(p) for p in sorted(selected_path.iterdir())
                    if p.suffix in {'.json','.safetensors'}} if spec['kind'] == 'adapter' else identity['base_files'])
inference_key = object_hash({'run_key':RUN_KEY,'selection':selection,'weights':weight_identity})
submission_dir = RUN / 'submissions' / inference_key[:16]
cache_dir = submission_dir / 'cache'; cache_dir.mkdir(parents=True, exist_ok=True)
test = pd.read_csv(ROOT / 'resources/Test.csv', dtype=str, keep_default_na=False)
template = pd.read_csv(ROOT / 'resources/SampleSubmission.csv', dtype=str, keep_default_na=False)
assert test.ID.is_unique and template.ID.is_unique and set(test.ID) == set(template.ID)
model = load_model(spec)
predictions = []
try:
    for image_id in tqdm(test.ID, desc='Test inference'):
        image_hash = file_hash(IMAGES / f'{image_id}.jpg')
        cached_path = cache_dir / f'{image_id}.json'
        cached = json.loads(cached_path.read_text()) if cached_path.exists() else None
        if cached and cached.get('inference_key') == inference_key and cached.get('image_hash') == image_hash:
            prediction = cached['prediction']
        else:
            details = predict(model, image_id, beams, return_details=True)
            assert not details['hit_token_limit'], f'Generation exhausted its token limit for {image_id}; inspect before submitting.'
            prediction = details['Prediction']
            write_json(cached_path, {'ID':image_id,'inference_key':inference_key,
                                     'image_hash':image_hash,'prediction':prediction})
        predictions.append({'ID':image_id,'Target':prediction})
finally:
    del model; gc.collect(); torch.cuda.empty_cache()
submission = template[['ID']].merge(pd.DataFrame(predictions), on='ID', validate='one_to_one', how='left')
assert submission.Target.notna().all()
blank_ids = submission.loc[submission.Target.str.strip().eq(''),'ID'].tolist()
assert not blank_ids, f'Blank predictions require investigation before submitting: {blank_ids[:10]}'
submission_path = submission_dir / 'submission_glm_curriculum.csv'
submission.to_csv(submission_path, index=False)
write_json(submission_dir / 'metadata.json', {'inference_key':inference_key,'selection':selection,'rows':len(submission)})
print('Saved:', submission_path)
display(submission.head())


Bounded verification completed on the local RTX 4060: all code cells compile and notebook schema validation passes; split/exclusions, deterministic sampling, guaranteed clean views, prompt/target masks and shared image tensors pass. The longest labeled response is 38 tokens including newline and EOS. A one-step Trainer check on two images produced finite nonzero gradients and actual parameter changes in vision LoRA, language LoRA and projector, followed by successful generation. The smoke test disabled warmup for that single update and saved no adapter. Neither full training stage was run. The original notebook's SHA-256 remained unchanged.

These checks establish execution compatibility, not an accuracy improvement. Run the revised recipe from base in its new v7 directory to measure the fixed holdout score. The three-epoch budgets now contain 2,303 Stage 1 and 2,565 Stage 2 optimizer steps per epoch with the current dataset and default five views.

The stopping/handoff update is checked separately with bounded CPU tests; neither training stage is executed to validate this update.
